In [14]:
#load necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
import joblib

In [15]:
#load the datasets
print("Loading the methylation and clinical datasets")
clin_df = pd.read_csv('../data/processed/clinical_kirc_train.csv')
meth_df = pd.read_parquet('../data/processed/methylation_kirc_train.parquet')

Loading the methylation and clinical datasets


In [16]:
print("This is the clinical dataset:")
print(clin_df)
print("----------------------------------")
print("This is the methylation dataset:")
print(meth_df)

This is the clinical dataset:
     Unnamed: 0            sample tissue_type.samples
0             5  TCGA-DV-5573-01A               Tumor
1             8  TCGA-BP-4795-01A               Tumor
2             9  TCGA-BP-4795-11A              Normal
3            26  TCGA-DV-5565-01A               Tumor
4            27  TCGA-CJ-4869-11A              Normal
..          ...               ...                 ...
280         940  TCGA-A3-3385-11A              Normal
281         941  TCGA-A3-3385-01A               Tumor
282         942  TCGA-B8-5552-01B               Tumor
283         946  TCGA-B0-4821-01A               Tumor
284         947  TCGA-B0-4821-11A              Normal

[285 rows x 3 columns]
----------------------------------
This is the methylation dataset:
Composite Element REF  cg00000108  cg00000236  cg00000292  cg00000321  \
index                                                                   
TCGA-B0-5108-01A         0.972311    0.909915    0.601544    0.283628   
TCGA-B0-570

In [17]:
#extract sample IDs from the index of the methylation dataset
meth_samples = meth_df.index.values

#extract the raw methylation values (all columns)
meth_features = meth_df.values

In [18]:
#scale the data to prepare it for PCA
print("Scaling data...")
scaler = StandardScaler()
meth_features_scaled = scaler.fit_transform(meth_features)

Scaling data...


In [ ]:
print("run PCA to determine optimal n by plotting the elbow plot per PC (cumulative explained variance)")
#ClaudeCode Ver. 1: search the FULL feasible rank (n_samples-1) instead of an arbitrary 100-component
#cap. That cap fell short of where 80% cumulative variance is actually reached for this dataset
#(n=112), so cumulative_variance >= 0.80 was False everywhere tested, and np.argmax silently returned
#index 0 (numpy's documented behavior for an all-False array, not an error) -- producing optimal_n=1
#again despite the "real" selection formula being used, just for a different, more subtle reason
max_components = len(meth_features_scaled) - 1
pca_test = PCA(n_components=max_components, random_state=67)
pca_test.fit(meth_features_scaled)

#plot the elbow plot (cumulative explained variance)
plt.figure(figsize=(10, 6))
cumulative_variance = np.cumsum(pca_test.explained_variance_ratio_)
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.title('PCA Explained Variance (Elbow Plot)')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid(True)
plt.axhline(y=0.80, color='r', linestyle='-') # Adding a red line for 80% variance threshold
plt.text(5, 0.82, '80% Variance Threshold', color='red')
plt.show()

# Obtain optimal n (components needed to explain at least 80% of variance)
#ClaudeCode Ver. 1: guard against the silent-failure mode above -- if the threshold is genuinely
#unreachable within the searched range, fail loudly instead of quietly returning optimal_n=1
if not (cumulative_variance >= 0.80).any():
    raise ValueError(
        f"80% variance threshold not reached within max_components={max_components} "
        f"(max cumulative variance: {cumulative_variance[-1]:.4f}). Increase max_components."
    )
optimal_n = np.argmax(cumulative_variance >= 0.80) + 1
print(f"Optimal n selected based on 80% explained variance: {optimal_n}")


In [20]:
#perform the final PCA reductions (in our case, we will do 2D, 3D, and optimal n)

print("Running PCA (n=2)...")
pca_2d_model = PCA(n_components=2, random_state=42)
pca_2d = pca_2d_model.fit_transform(meth_features_scaled)

print("Running PCA (n=3)...")
pca_3d_model = PCA(n_components=3, random_state=42)
pca_3d = pca_3d_model.fit_transform(meth_features_scaled)

print(f"Running PCA (n={optimal_n})...")
pca_opt_model = PCA(n_components=optimal_n, random_state=42)
pca_opt = pca_opt_model.fit_transform(meth_features_scaled)

Running PCA (n=2)...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: invalid value encountered in matmul

Running PCA (n=3)...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: invalid value encountered in matmul

Running PCA (n=1)...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: invalid value encountered in matmul

In [ ]:
#perform K-means, Hierarchical, and GMM clustering on the PCA-reduced data (2D, 3D, and optimal n) and raw data

N_CLUSTERS_2 = 2 #we enforce 2 clusters since our ground truth is binary (tumor vs normal)

#feature 1: K-means on PCA 2d data
print("Running K-Means on PCA 2d data...")
kmeans_pca_2d = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_pca_2d_labels_k2 = kmeans_pca_2d.fit_predict(pca_2d)

#feature 2: K-means on PCA 3d data
print("Running K-Means on PCA 3d data...")
kmeans_pca_3d = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_pca_3d_labels_k2 = kmeans_pca_3d.fit_predict(pca_3d)

#feature 3: K-means on PCA optimal n data
print(f"Running K-Means on PCA optimal ({optimal_n}d) data...")
kmeans_pca_opt = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_pca_opt_labels_k2 = kmeans_pca_opt.fit_predict(pca_opt)

#feature 4: K-means on Raw data
print("Running K-Means on Raw data (This may take a while)...")
kmeans_raw = KMeans(n_clusters=N_CLUSTERS_2, random_state=42)
kmeans_raw_labels_k2 = kmeans_raw.fit_predict(meth_features_scaled)

##############

#feature 5: Hierarchical on PCA 2d data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on PCA 2d data...")
hierarchical_pca_2d = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_pca_2d_labels_k2 = hierarchical_pca_2d.fit_predict(pca_2d)

# feature 6: Hierarchical on PCA 3d data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on PCA 3d data...")
hierarchical_pca_3d = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_pca_3d_labels_k2 = hierarchical_pca_3d.fit_predict(pca_3d)

#feature 7: Hierarchical on PCA optimal n data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on PCA optimal ({optimal_n}d) data...")
hierarchical_pca_opt = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_pca_opt_labels_k2 = hierarchical_pca_opt.fit_predict(pca_opt)

#feature 8: Hierarchical on Raw data
print(f"Running Hierarchical Clustering (K={N_CLUSTERS_2}) on Raw data...")
hierarchical_raw = AgglomerativeClustering(n_clusters=N_CLUSTERS_2, linkage='ward')
hierarchical_raw_labels_k2 = hierarchical_raw.fit_predict(meth_features_scaled)

#################

#feature 9: GMM on PCA 2d data
print(f"Running GMM (n_components={N_CLUSTERS_2}) on PCA 2d data...")
gmm_pca_2d = GaussianMixture(n_components=N_CLUSTERS_2, covariance_type='full',random_state=42, n_init=20)
gmm_pca_2d_labels_k2 = gmm_pca_2d.fit_predict(pca_2d)

#feature 10: GMM on PCA 3d data
print(f"Running GMM (n_components={N_CLUSTERS_2}) on PCA 3d data...")
gmm_pca_3d = GaussianMixture(n_components=N_CLUSTERS_2, covariance_type='full', random_state=42, n_init=20)
gmm_pca_3d_labels_k2 = gmm_pca_3d.fit_predict(pca_3d)

#feature 11: GMM on PCA optimal n data
#ClaudeCode Ver. 1: use covariance_type='diag' here instead of 'full'. A full covariance matrix in
#optimal_n (112) dimensions needs ~n*(n+1)/2 = 6,328 parameters per component (~12,656 total) from
#only ~285 training samples -- severely underdetermined. This let the model fit training data almost
#perfectly (ARI 0.8893) while completely failing to generalize: on holdout data it collapsed to
#assigning every single sample to one component (posterior probability exactly 1.0/0.0), confirmed via
#05_holdout.ipynb. 'diag' needs only n=112 parameters per component, matching the low-dimensional (2d,
#3d) cases' parameter-to-sample ratio much more closely and avoiding this overfitting failure mode.
print(f"Running GMM (n_components={N_CLUSTERS_2}) on PCA optimal ({optimal_n}d) data...")
gmm_pca_opt = GaussianMixture(n_components=N_CLUSTERS_2, covariance_type='diag', random_state=42, n_init=20)
gmm_pca_opt_labels_k2 = gmm_pca_opt.fit_predict(pca_opt)

#we omit GMM on raw data for now due to memory issues; if you want to run it, uncomment the following lines and ensure you have enough memory available.

# # Feature 12: GMM on Raw data
# print(f"Running GMM (n_components={N_CLUSTERS_2}) on Raw data (Warning: This may take a while or use high memory)...")
# # Note: if it fails due to memory, you can add covariance_type='diag' to the GMM instantiation
# gmm_raw = GaussianMixture(n_components=N_CLUSTERS_2, covariance_type='full', random_state=42)
# gmm_raw_labels_k2 = gmm_raw.fit_predict(meth_features_scaled)

In [22]:
#compile the new cluster features into a dataframe
print("Compiling results...")
cluster_results = pd.DataFrame({
    'sample': meth_samples,
    'kmeans_pca_2_k2': kmeans_pca_2d_labels_k2,
    'kmeans_pca_3_k2': kmeans_pca_3d_labels_k2,
    'kmeans_pca_opt_k2': kmeans_pca_opt_labels_k2,
    'kmeans_raw_k2': kmeans_raw_labels_k2,
    'hierarchical_pca_2_k2': hierarchical_pca_2d_labels_k2,
    'hierarchical_pca_3_k2': hierarchical_pca_3d_labels_k2,
    'hierarchical_pca_opt_k2': hierarchical_pca_opt_labels_k2,
    'hierarchical_raw_k2': hierarchical_raw_labels_k2,
    'gmm_pca_2_k2': gmm_pca_2d_labels_k2,
    'gmm_pca_3_k2': gmm_pca_3d_labels_k2,
    'gmm_pca_opt_k2': gmm_pca_opt_labels_k2
    # 'gmm_raw_k2': gmm_raw_labels_k2
})

#merge with the clinical dataset
print("Merging with clinical data...")
final_df = pd.merge(clin_df, cluster_results, on='sample', how='left')

Compiling results...
Merging with clinical data...


In [23]:
print("This is the final dataset with clustering features merged with clinical data:")
print(final_df)

This is the final dataset with clustering features merged with clinical data:
     Unnamed: 0            sample tissue_type.samples  kmeans_pca_2_k2  \
0             5  TCGA-DV-5573-01A               Tumor                0   
1             8  TCGA-BP-4795-01A               Tumor                1   
2             9  TCGA-BP-4795-11A              Normal                1   
3            26  TCGA-DV-5565-01A               Tumor                0   
4            27  TCGA-CJ-4869-11A              Normal                1   
..          ...               ...                 ...              ...   
280         940  TCGA-A3-3385-11A              Normal                1   
281         941  TCGA-A3-3385-01A               Tumor                0   
282         942  TCGA-B8-5552-01B               Tumor                0   
283         946  TCGA-B0-4821-01A               Tumor                0   
284         947  TCGA-B0-4821-11A              Normal                1   

     kmeans_pca_3_k2  kmeans_pca_

In [24]:
# 6. Save the new dataset
output_file = '../data/processed/clinical_kirc_train_clustered_pca.csv'
final_df.to_csv(output_file, index=False)
print(f"Process complete! Saved new dataset to {output_file}")

Process complete! Saved new dataset to ../data/processed/clinical_kirc_train_clustered_pca.csv


In [ ]:
#define columns to check ARI scores against the ground truth tissue type labels
cluster_columns = [
    'kmeans_pca_2_k2',
    'kmeans_pca_3_k2',
    'kmeans_pca_opt_k2',
    'kmeans_raw_k2',
    'hierarchical_pca_2_k2',
    'hierarchical_pca_3_k2',
    'hierarchical_pca_opt_k2',
    'hierarchical_raw_k2',
    'gmm_pca_2_k2',
    'gmm_pca_3_k2',
    'gmm_pca_opt_k2'
    # 'gmm_raw_k2'
]

#set the target column
target_column = 'tissue_type.samples'

columns_to_check = [target_column] + cluster_columns
valid_df = final_df.dropna(subset=columns_to_check).copy()

#extract the true labels
true_labels = valid_df[target_column]

#compute and print the ARI for each feature
print(f"Adjusted Rand Index (ARI) Scores (Evaluated on {len(valid_df)} samples):")
print("-" * 55)

#ClaudeCode Ver. 1: store every column's ARI so the best one can be selected automatically below,
#instead of eyeballing this printout and hardcoding a column name in 04_classification.ipynb
ari_scores = {}
for col in cluster_columns:
    pred_labels = valid_df[col]
    ari_score = adjusted_rand_score(true_labels, pred_labels)
    ari_scores[col] = ari_score
    print(f"{col:<25} : {ari_score:.4f}")

#the single highest-ARI column, regardless of algorithm -- this is what 04_classification.ipynb
#should train pseudo-label classifiers on, since it only needs labels for the already-fit training set
best_pseudo_label_col = max(ari_scores, key=ari_scores.get)
print(f"\nBest pseudo-label column overall: {best_pseudo_label_col} (ARI={ari_scores[best_pseudo_label_col]:.4f})")

with open('../models/best_pseudo_label_column.txt', 'w') as f:
    f.write(best_pseudo_label_col)


In [ ]:
#ClaudeCode Ver. 1: label-free (unsupervised) clustering quality metrics, to complement ARI -- ARI
#needs ground truth, silhouette/Davies-Bouldin do not, which matters for framing this as a genuinely
#unsupervised pipeline (a reviewer could ask "how would you choose a clustering without labels?").
#Computed directly on the original (label array, feature matrix) pairs from cell 200bce30, NOT via
#valid_df -- valid_df's row order follows the clinical CSV (from the clin_df merge), while the label
#arrays and pca_2d/pca_3d/pca_opt/meth_features_scaled all follow the methylation parquet's row order;
#mixing the two would silently misalign labels and features (the same class of bug fixed earlier
#today in 04_classification.ipynb and 05_holdout.ipynb).
from sklearn.metrics import silhouette_score, davies_bouldin_score

label_feature_registry = {
    'kmeans_pca_2_k2': (kmeans_pca_2d_labels_k2, pca_2d),
    'hierarchical_pca_2_k2': (hierarchical_pca_2d_labels_k2, pca_2d),
    'gmm_pca_2_k2': (gmm_pca_2d_labels_k2, pca_2d),
    'kmeans_pca_3_k2': (kmeans_pca_3d_labels_k2, pca_3d),
    'hierarchical_pca_3_k2': (hierarchical_pca_3d_labels_k2, pca_3d),
    'gmm_pca_3_k2': (gmm_pca_3d_labels_k2, pca_3d),
    'kmeans_pca_opt_k2': (kmeans_pca_opt_labels_k2, pca_opt),
    'hierarchical_pca_opt_k2': (hierarchical_pca_opt_labels_k2, pca_opt),
    'gmm_pca_opt_k2': (gmm_pca_opt_labels_k2, pca_opt),
    'kmeans_raw_k2': (kmeans_raw_labels_k2, meth_features_scaled),
    'hierarchical_raw_k2': (hierarchical_raw_labels_k2, meth_features_scaled),
}

print(f"\nLabel-free clustering quality metrics (no ground truth needed):")
print("-" * 70)
print(f"{'Column':<25} {'Silhouette':>12} {'Davies-Bouldin':>16}")
for col in cluster_columns:
    labels, X_for_metric = label_feature_registry[col]
    if len(set(labels)) < 2:
        print(f"{col:<25} {'N/A (1 cluster)':>28}")
        continue
    sil = silhouette_score(X_for_metric, labels)
    db = davies_bouldin_score(X_for_metric, labels)
    print(f"{col:<25} {sil:>12.4f} {db:>16.4f}")


In [ ]:
#ClaudeCode Ver. 1: automatically select and save whichever PCA+clustering configuration scored
#highest ARI, restricted to algorithms that support .predict() on new/unseen data. This matters for
#05_holdout.ipynb, which needs to score the held-out set with the saved clustering model --
#AgglomerativeClustering (hierarchical) has no such method (it only supports fit_predict on the data
#it was fit on), so even if a hierarchical_*_k2 column scores highest ARI overall, it's excluded here.
#Raw-feature clustering is also excluded since this pool is specifically "PCA model + clustering model"
#pairs that 05_holdout.ipynb can apply end-to-end to new data.
predictable_model_registry = {
    'kmeans_pca_2_k2': (pca_2d_model, kmeans_pca_2d),
    'kmeans_pca_3_k2': (pca_3d_model, kmeans_pca_3d),
    'kmeans_pca_opt_k2': (pca_opt_model, kmeans_pca_opt),
    'gmm_pca_2_k2': (pca_2d_model, gmm_pca_2d),
    'gmm_pca_3_k2': (pca_3d_model, gmm_pca_3d),
    'gmm_pca_opt_k2': (pca_opt_model, gmm_pca_opt),
}

predictable_ari_scores = {col: ari_scores[col] for col in predictable_model_registry}
best_predictable_col = max(predictable_ari_scores, key=predictable_ari_scores.get)
best_pca_model, best_clustering_model = predictable_model_registry[best_predictable_col]

print(f"Best predictable (PCA + KMeans/GMM) configuration: {best_predictable_col} "
      f"(ARI={predictable_ari_scores[best_predictable_col]:.4f})")

#save the best PCA model
joblib.dump(best_pca_model, '../models/pca_best_model.joblib')

#save best clustering
joblib.dump(best_clustering_model, '../models/clustering_best_model.joblib')
